### Histogram Aggregator

Geometric Regularization from Overparameterized Learning Explains Double Descent and other findings

This noteboook is template for what will be provided with supplemental material

In [ ]:
!pip install Automunge

In [ ]:
#automunge import

from Automunge import *
am = AutoMunge()

In [ ]:
#other imports

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

import matplotlib.pyplot as plt
# import numpy as np
from matplotlib import colors
from matplotlib.ticker import PercentFormatter

import timeit
import pickle


In [ ]:
#load titanic data set
ds = tfds.load('titanic', split='train', shuffle_files=False, with_info=True)
df_train = tfds.as_dataframe(ds[0], ds[1])
columns = list(df_train)
#this converts column headers to remove a prefix
columns.remove('survived')
columns = [x[9:] for x in columns]
columns += ['survived']
df_train.columns = columns

labels_column = 'survived'

df_train.head()

In [ ]:
#this function initializes dense network model based on specified width, depth, initialization
#init_param1 / init_param2 are min/max for uniform initialization (RandomUniform)
#init_param3 is scale for normal (RandomNormal)
#returns one initalized model

def create_model(width,
                 depth,
                 init='uniform',
                 init_param1=-0.5,
                 init_param2=0.5,
                 init_param3=0.5):

  model = False

  #for init = 'uniform', param1 and param2 are min and max
  #for init = 'normal', param3 is scale and param1 and 2 ignored

  if init == 'uniform':
    initializer = tf.keras.initializers.RandomUniform(
      minval=init_param1, maxval=init_param2, seed=None
    )
  elif init == 'normal':
    initializer = tf.keras.initializers.RandomNormal(mean=0., stddev=init_param3)

  elif init == 'HeNormal':
    initializer = tf.keras.initializers.HeNormal(seed=None)

  elif init == 'HeUniform':
    initializer = tf.keras.initializers.HeNormal(seed=None)

  #create model with keras
  model = Sequential()

  for i in range(depth):
    model.add(Dense(width, activation='relu', kernel_initializer=initializer))

    # #batch norm inclusion scenario
    # model.add(Dense(width, activation=None, kernel_initializer=initializer))
    # model.add(BatchNormalization())
    # model.add(Activation("relu"))

    # #tanh smooth activation function scenario
    # #applying tanh to evaluate impact of a smooth activation function
    # model.add(Dense(width, activation='tanh', kernel_initializer=initializer))

  model.add(Dense(1, activation='sigmoid', kernel_initializer=initializer))

  #compile model
  model.compile(loss='binary_crossentropy', optimizer='adam', metrics=["accuracy"])

  return model

In [ ]:
#this operation encodes the features

#let's try with just the three features
#these features were selected based on a feature importance evaluation
df_train = df_train[['pclass', 'sex', 'age', labels_column]]

#ordl is ordinal encoding, bnry is boolean integer, nmbr is z-score normalization
assigncat = {'ordl':'pclass',
             'bnry':['sex', labels_column],
             'nmbr':'age'}

train_am, train_ID, labels_am, \
val, val_ID, val_labels, \
test, test_ID, test_labels, \
postprocess_dict = \
am.automunge(df_train,
            labels_column = labels_column,
            assigncat = assigncat,
            MLinfill=False,
            NArw_marker=False,
            shuffletrain=False,
            pandasoutput = True,
            printstatus=False)

train_am.head()

In [ ]:

#these are the scenarios we'll cycle through,
#each combination will be one experiment
#as configured init_params don't accept lists
scenarios = \
{'width' : [3,6,9,12,15,18],
'depth' : [1,2,3,4,5,6],
'init' : ['HeNormal', 'HeUniform', 'normal', 'uniform'],
'init_param1' : -1,
'init_param2' : 1,
'init_param3' : 0.5,
'row_address' : [(0,50),(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8)],
}

#row address refers to rows inspected. e.g. (0,50) is for train_am[0:50], (0,1) for train_am[0:1]
#this notebook only looks at target index in row_address list,
#e.g. target_row_address_index = {1,2} looks at (0,1),(1,2)

# target_row_address_index = {0}
target_row_address_index = {1,2}
# target_row_address_index = {3,4}
# target_row_address_index = {5,6}
# target_row_address_index = {7,8}

#sample_count is the number of samples collected in each histogram
#in another notebook we based this on time span instead of quantity
sample_count = 2500

experiment_log = {}
experiment_number = 0
max_experiment_number = 0

#we'll store the collected samples serving as basis for histogram in this dataframe
#which we'll download at conclusion of notebook, one column per histogram
#the first 13 rows store collected statistics for the experiment
experiment_df = pd.DataFrame(index = ['chart_title', 'experiment_number', 'row_address', 'init', 'width', 'depth', 'sample_count', 'min', 'max', 'median', 'mean', 'stdev', 'mad'] + list(range(sample_count)))


for row_address in scenarios['row_address']:

  # print()
  # print("____________________________________")
  # print("************************************")
  # print()
  # print("row_address scenario = ", row_address)
  # print()

  train = tf.convert_to_tensor(train_am[row_address[0]:row_address[1]])
  # labels = tf.convert_to_tensor(labels)
  labels = labels_am[row_address[0]:row_address[1]].to_numpy()

  for init in scenarios['init']:

    # print()
    # print("____________________________________")
    # print("************************************")
    # print()
    # print("init scenario = ", init)
    # print()

    for width in scenarios['width']:

      for depth in scenarios['depth']:

        #we'll run different notebooks for different row_address scenarios
        if scenarios['row_address'].index(row_address) in target_row_address_index:

          print()
          print("experiment_number: ", experiment_number)
          print("row_address: ", row_address)
          print("init: ", init)
          print("width: ", width)
          print("depth: ", depth)
          print()

          #chart_title is the string at top of each histogram detailing configuration
          chart_title = 'row_address:' + str(row_address) + ' init:' + str(init) \
                        + ' width:'+str(width) + ' depth:'+str(depth) \
                        + ' sample_count:' + str(sample_count) \
                        + ' exp#:'+str(experiment_number)

          start_time = timeit.default_timer()

          #we'll aggregate sampled loss values in a unique list for each setup
          loss_list = []

          #here is where we collect samples
          for i in range(sample_count):

            #initialize model (the sampled weights are conducted with initialization)
            model = create_model(width,
                                depth,
                                init=init,
                                init_param1=scenarios['init_param1'],
                                init_param2=scenarios['init_param2'],
                                init_param3=scenarios['init_param3'])

            #run inference without training
            predictions = model.predict(train)

            #calucate binary cross entropy and append to list
            cce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
            loss = cce(labels, predictions).numpy()
            loss_list.append(loss)

          #now store those experiment results in the dataframe
          stats_df = pd.DataFrame({'loss_list':loss_list})
          experiment_df[experiment_number] = [chart_title, experiment_number, row_address, init, width, depth, sample_count,
                                              stats_df['loss_list'].min(),
                                              stats_df['loss_list'].max(),
                                              stats_df['loss_list'].median(),
                                              stats_df['loss_list'].mean(),
                                              stats_df['loss_list'].std(),
                                              stats_df['loss_list'].mad(),
                                              ] + loss_list

          # stats_df = pd.DataFrame({'loss_list':loss_list})
          print("min = ", stats_df['loss_list'].min())
          print("max = ", stats_df['loss_list'].max())
          print("median = ", stats_df['loss_list'].median())
          print("mean = ", stats_df['loss_list'].mean())
          # print("mode = ", stats_df['loss_list'].mode())
          print("stdev = ", stats_df['loss_list'].std())
          print("mad = ", stats_df['loss_list'].mad())
          print()

          #now we'll display the histogram in printouts for this cell
          fig, axs = plt.subplots(1, sharey=False, tight_layout=True)

          n, bins, patches = \
          axs.hist(np.array(loss_list), bins=np.arange(np.array(loss_list).min(), np.array(loss_list).max() + 0.02, 0.02), density=True)

          # axs.plot(bins)
          axs.set_xlabel('Loss')
          axs.set_ylabel('density')
          axs.set_title(chart_title)

          filename = chart_title + '_122221' + '.png'

          #I think this save operation doesn't work in colab, so just download the dataframe below
          # plt.savefig(filename, bbox_inches='tight')

          plt.show()

          print()
          print("time elapsed:")
          print(timeit.default_timer() - start_time)
          print()

          max_experiment_number = experiment_number

        experiment_number += 1

print("complete")

In [ ]:
#the first 13 rows are the statistics for each scenario
experiment_df[:14]

In [ ]:
#this cell downloads the log of samples from colaboratory

experiment_download_name = '_exp_' + str(max_experiment_number) + '_123021' + '_colab_row_addres_' + str(row_address) + '.csv'

experiment_df.to_csv(experiment_download_name, index=False)

from google.colab import files

files.download(experiment_download_name)


In [ ]:
#voila

In [ ]:
#this is an alternate configuration that collects samples
#based on sampling time instead of number of samples
#this is useful when trying to max out colab running window
#e.g. can collect one configuration over 23.5 hours
#by using:
#while (timeit.default_timer() - start_time) < (24*60*60 - 30*60):

In [ ]:


# #initialize model
# scenarios = \
# {'width' : [18], #[3,6,9,12,15,18],
# 'depth' : [3], #[1,2,3,4,5,6],
# 'init' : ['HeNormal'], #['HeNormal', 'HeUniform', 'normal', 'uniform'],
# 'init_param1' : -1,
# 'init_param2' : 1,
# 'init_param3' : 0.5,
# 'row_address' : [(0,1)], #[(0,50),(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8)],
# }

# target_row_address_index = {0}
# # target_row_address_index = {1,2}
# # target_row_address_index = {3,4}
# # target_row_address_index = {5,6}
# # target_row_address_index = {7,8}

# sample_count = '24hr'

# experiment_log = {}
# experiment_number = 68
# max_experiment_number = 0

# index_list = ['chart_title', 'experiment_number', 'row_address', 'init', 'width', 'depth', 'sample_count', 'min', 'max', 'median', 'mean', 'stdev', 'mad']
# experiment_df = pd.DataFrame(index = index_list)

# for row_address in scenarios['row_address']:

#   # print()
#   # print("____________________________________")
#   # print("************************************")
#   # print()
#   # print("row_address scenario = ", row_address)
#   # print()

#   train = tf.convert_to_tensor(train_am[row_address[0]:row_address[1]])
#   # labels = tf.convert_to_tensor(labels)
#   labels = labels_am[row_address[0]:row_address[1]].to_numpy()

#   for init in scenarios['init']:

#     # print()
#     # print("____________________________________")
#     # print("************************************")
#     # print()
#     # print("init scenario = ", init)
#     # print()

#     for width in scenarios['width']:

#       for depth in scenarios['depth']:

#         #we'll run different notebooks for different row_address scenarios
#         if scenarios['row_address'].index(row_address) in target_row_address_index:

#           print()
#           print("experiment_number: ", experiment_number)
#           print("row_address: ", row_address)
#           print("init: ", init)
#           print("width: ", width)
#           print("depth: ", depth)
#           print()

#           chart_title = 'row_address:' + str(row_address) + ' init:' + str(init) \
#                         + ' width:'+str(width) + ' depth:'+str(depth) \
#                         + ' sample_count:' + str(sample_count) \
#                         + ' exp#:'+str(experiment_number)

#           start_time = timeit.default_timer()

#           loss_list = []
#           # for i in range(sample_count):
#           i = -1
#           while (timeit.default_timer() - start_time) < (24*60*60 - 30*60):
#             i+= 1

#             model = create_model(width,
#                                 depth,
#                                 init=init,
#                                 init_param1=scenarios['init_param1'],
#                                 init_param2=scenarios['init_param2'],
#                                 init_param3=scenarios['init_param3'])

#             predictions = model.predict(train)
#             cce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
#             loss = cce(labels, predictions).numpy()
#             loss_list.append(loss)

#           stats_df = pd.DataFrame({'loss_list':loss_list})
#           experiment_df2 = pd.DataFrame({experiment_number : [chart_title, experiment_number, row_address, init, width, depth, sample_count,
#                                               stats_df['loss_list'].min(),
#                                               stats_df['loss_list'].max(),
#                                               stats_df['loss_list'].median(),
#                                               stats_df['loss_list'].mean(),
#                                               stats_df['loss_list'].std(),
#                                               stats_df['loss_list'].mad(),
#                                               ] + loss_list}, index = index_list + list(range(len(loss_list))))

#           experiment_df = pd.concat([experiment_df, experiment_df2], axis=1)

#           # stats_df = pd.DataFrame({'loss_list':loss_list})
#           print("min = ", stats_df['loss_list'].min())
#           print("max = ", stats_df['loss_list'].max())
#           print("median = ", stats_df['loss_list'].median())
#           print("mean = ", stats_df['loss_list'].mean())
#           # print("mode = ", stats_df['loss_list'].mode())
#           print("stdev = ", stats_df['loss_list'].std())
#           print("mad = ", stats_df['loss_list'].mad())
#           print()

#           fig, axs = plt.subplots(1, sharey=False, tight_layout=True)

#           n, bins, patches = \
#           axs.hist(np.array(loss_list), bins=np.arange(np.array(loss_list).min(), np.array(loss_list).max() + 0.02, 0.02), density=True)

#           # axs.plot(bins)
#           axs.set_xlabel('Loss')
#           axs.set_ylabel('density')
#           axs.set_title(chart_title)

#           filename = chart_title + '_010822_Fig3_scale' + '.png'

#           # plt.savefig(filename, bbox_inches='tight')

#           plt.show()

#           print()
#           print("time elapsed:")
#           print(timeit.default_timer() - start_time)
#           print()

#           max_experiment_number = experiment_number

#         experiment_number += 1

# print("complete")